# BeeSpace 05 - Machine Learning para risco da colmeia

MVP de classificação de risco usando Random Forest. A base é sintética e representa a integração entre dados Copernicus e sinais locais da colmeia.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

OUTPUT_DIR = Path("outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.inspection import permutation_importance
import joblib

In [ ]:
def gerar_dados_sinteticos(n=1500):
    dados = pd.DataFrame({
        "ndvi": np.random.uniform(0.15, 0.85, n),
        "evi": np.random.uniform(0.10, 0.75, n),
        "ndwi": np.random.uniform(-0.30, 0.55, n),
        "temp_media": np.random.uniform(10, 40, n),
        "precipitacao_7d": np.random.uniform(0, 120, n),
        "umidade_solo": np.random.uniform(0.05, 0.70, n),
        "poluicao_indice": np.random.uniform(0, 1, n),
        "perc_mata_nativa": np.random.uniform(0, 1, n),
        "perc_agricultura": np.random.uniform(0, 1, n),
        "perc_solo_exposto": np.random.uniform(0, 0.7, n),
        "temp_colmeia": np.random.uniform(25, 42, n),
        "umidade_colmeia": np.random.uniform(35, 85, n),
        "variacao_peso_7d": np.random.uniform(-4, 5, n),
        "atividade_abelhas": np.random.uniform(0, 1, n),
        "anomalia_acustica": np.random.uniform(0, 1, n),
        "mortalidade_observada": np.random.uniform(0, 1, n),
    })

    soma = dados[["perc_mata_nativa", "perc_agricultura", "perc_solo_exposto"]].sum(axis=1)
    for col in ["perc_mata_nativa", "perc_agricultura", "perc_solo_exposto"]:
        dados[col] = dados[col] / soma

    score = (
        (dados["ndvi"] < 0.35).astype(int) * 2
        + (dados["evi"] < 0.25).astype(int)
        + (dados["ndwi"] < 0.00).astype(int)
        + (dados["precipitacao_7d"] < 15).astype(int)
        + (dados["umidade_solo"] < 0.20).astype(int)
        + (dados["temp_media"] > 33).astype(int)
        + (dados["temp_colmeia"] > 37).astype(int) * 2
        + (dados["umidade_colmeia"] < 45).astype(int)
        + (dados["variacao_peso_7d"] < -1.0).astype(int) * 2
        + (dados["atividade_abelhas"] < 0.35).astype(int) * 2
        + (dados["anomalia_acustica"] > 0.65).astype(int) * 2
        + (dados["mortalidade_observada"] > 0.55).astype(int) * 3
        + (dados["poluicao_indice"] > 0.70).astype(int)
        + (dados["perc_agricultura"] > 0.60).astype(int)
        + (dados["perc_solo_exposto"] > 0.25).astype(int)
        + (dados["perc_mata_nativa"] < 0.20).astype(int)
    )

    dados["classe_risco"] = np.select(
        [score <= 3, (score > 3) & (score <= 7), score > 7],
        ["normal", "atencao", "alerta"],
        default="atencao"
    )
    return dados

dados = gerar_dados_sinteticos()
dados.head()

In [ ]:
X = dados.drop(columns=["classe_risco"])
y = dados["classe_risco"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=RANDOM_STATE, stratify=y
)

modelo = Pipeline(steps=[
    ("scaler", StandardScaler()),
    ("classifier", RandomForestClassifier(
        n_estimators=300,
        max_depth=8,
        random_state=RANDOM_STATE,
        class_weight="balanced"
    ))
])

modelo.fit(X_train, y_train)
y_pred = modelo.predict(X_test)

print("Matriz de confusão:")
print(confusion_matrix(y_test, y_pred))
print("\nRelatório de classificação:")
print(classification_report(y_test, y_pred))

In [ ]:
resultado = permutation_importance(
    modelo, X_test, y_test, n_repeats=10, random_state=RANDOM_STATE, scoring="accuracy"
)

importancia = pd.DataFrame({
    "variavel": X_test.columns,
    "importancia_media": resultado.importances_mean,
    "desvio": resultado.importances_std
}).sort_values("importancia_media", ascending=False)

importancia.head(12)

In [ ]:
top = importancia.head(10).sort_values("importancia_media")
plt.figure(figsize=(8, 5))
plt.barh(top["variavel"], top["importancia_media"])
plt.title("Principais variáveis para a classificação")
plt.xlabel("Importância média por permutação")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "importancia_variaveis_ml_beespace.png", dpi=150)
plt.show()

In [ ]:
nova_colmeia = pd.DataFrame([{
    "ndvi": 0.31,
    "evi": 0.28,
    "ndwi": -0.08,
    "temp_media": 34.5,
    "precipitacao_7d": 8.0,
    "umidade_solo": 0.16,
    "poluicao_indice": 0.74,
    "perc_mata_nativa": 0.15,
    "perc_agricultura": 0.72,
    "perc_solo_exposto": 0.13,
    "temp_colmeia": 38.2,
    "umidade_colmeia": 42.0,
    "variacao_peso_7d": -2.4,
    "atividade_abelhas": 0.22,
    "anomalia_acustica": 0.81,
    "mortalidade_observada": 0.62,
}])

classe = modelo.predict(nova_colmeia)[0]
probs = modelo.predict_proba(nova_colmeia)[0]
classes = modelo.named_steps["classifier"].classes_

print(f"Classe prevista: {classe}")
for c, p in zip(classes, probs):
    print(f"{c}: {p:.2%}")

In [ ]:
joblib.dump(modelo, OUTPUT_DIR / "modelo_beespace_mvp.pkl")
dados.to_csv(OUTPUT_DIR / "dados_sinteticos_beespace.csv", index=False)
print("Artefatos salvos na pasta outputs.")